# Customer Retention Prediction

The **objective** of this notebook is to develop and evaluate a machine
learning model that predicts customer retention using historical
purchasing behavior.

The analysis will evaluate model performance, identify the historical
behaviors associated with retention, and generate customer-level
retention risk predictions that can support targeted retention
strategies.
```
Retained = 1
→ customer purchased again during Sep–Nov 2011

Retained = 0
→ customer did not purchase during Sep–Nov 2011
→ treated as non-retained / churn-risk
```

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

retention_file = PROCESSED_DIR / "retention_modeling_data.csv"

retention_df = pd.read_csv(retention_file)

print(f"Retention dataset loaded: {retention_df.shape}")
retention_df.head()

Retention dataset loaded: (3317, 6)


,CustomerID,Historical_Orders,Historical_Recency,Customer_Tenure,Future_Orders,Retained
0,12346.0,1,225,0,0,0
1,12347.0,5,29,238,1,1
2,12348.0,3,148,110,1,1
3,12350.0,1,210,0,0,0
4,12352.0,5,162,34,3,1


### Define Predictors and Target

Only customer behavior observed before the prediction cutoff is used
as model input.

The target variable represents whether the customer was retained
during the subsequent September–November 2011 outcome period.

In [3]:
feature_cols = [
    "Historical_Orders",
    "Historical_Recency",
    "Customer_Tenure"
]

X = retention_df[feature_cols]
y = retention_df["Retained"]

print("Features :", X.columns.tolist())

print("\nTarget :", y.name)

Features : ['Historical_Orders', 'Historical_Recency', 'Customer_Tenure']

Target : Retained


### Train-Test Split

The training data is used to fit the model, while the testing data is
held back for evaluating how well the model generalizes to unseen
customers.

```X_train + y_train``` → learn

```X_test``` → predict

```y_test``` → evaluate

- Retained (1)    =  56.41%
- Not retained (0)  = 43.59%
```stratify=y``` keeps approximately that same proportion in both the training and testing sets for balanced learning.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,  # use 20% of the data for testing
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training set: (2653, 3)
Testing set: (664, 3)

Training target distribution:
Retained
1    56.39
0    43.61
Name: proportion, dtype: float64

Testing target distribution:
Retained
1    56.48
0    43.52
Name: proportion, dtype: float64


### Feature Scaling
Our three predictors have very different ranges.
So, the variables are standardized before Logistic Regression to bring them on a comparable scale.

The scaler is fitted only on the training data and then applied to both
the training and testing data to prevent data leakage.
```
                        z= (X-μ) / σ
                        where μ=0, σ=1

fit
 ↓
Learn mean and standard deviation

transform
 ↓
Use them to scale the data
```

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

Scaled training shape: (2653, 3)
Scaled testing shape: (664, 3)


### Logistic Regression Model

A Logistic Regression model is trained to predict whether a customer
will be retained during the future observation period.

In [6]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(random_state=42, max_iter=1000)

logistic_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [7]:
# y_pred → predicted class (0 or 1)
# y_pred_proba → probability that the customer will be retained (class 1)

y_pred = logistic_model.predict(X_test_scaled)
y_pred_proba = logistic_model.predict_proba(X_test_scaled)[:, 1]

print("Predictions generated.")
print("Predicted classes:", y_pred[:10])
print("Predicted probabilities:", y_pred_proba[:10])

Predictions generated.
Predicted classes: [1 0 0 1 0 1 0 1 0 0]
Predicted probabilities: [0.83369579 0.48324658 0.43572532 0.72835755 0.45799524 0.6710134
 0.40282722 0.79298362 0.47574993 0.36554043]


In [8]:
# The model is evaluated on the held-out testing set using classification metrics and ROC-AUC.

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
                    # (actual val, predicted val)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

Accuracy:  0.6822
Precision: 0.7531
Recall:    0.6507
F1 Score:  0.6981
ROC-AUC:   0.7392

Confusion Matrix:
[[209  80]
 [131 244]]


For the retained class:
- 244 retained customers were correctly identified.
- 131 retained customers were incorrectly classified as churn-risk.
- 209 churn-risk customers were correctly identified.
- 80 churn-risk customers were incorrectly classified as retained.

For churn-risk class:
- precision ≈ 61.47%
- recall ≈ 72.32%

That means the model catches about 72% of the customers who actually become non-retained.

In [9]:
from sklearn.metrics import classification_report

# Summary of how well your classification model performed
print(classification_report(
    y_test,
    y_pred,
    target_names=["Churn-risk (0)", "Retained (1)"]
))

                precision    recall  f1-score   support

Churn-risk (0)       0.61      0.72      0.66       289
  Retained (1)       0.75      0.65      0.70       375

      accuracy                           0.68       664
     macro avg       0.68      0.69      0.68       664
  weighted avg       0.69      0.68      0.68       664



### Logistic Regression Coefficients

They are examined to understand how historical customer behavior is associated with the probability of retention.

Positive coefficients indicate a higher predicted probability of
retention, while negative coefficients indicate a lower predicted
probability of retention.

In [10]:
coefficient_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": logistic_model.coef_[0]
})

coefficient_df

,Feature,Coefficient
0,Historical_Orders,1.254870
1,Historical_Recency,-0.288053
2,Customer_Tenure,0.238851


- Customers with higher Historical_Orders tend to have a higher predicted probability of retention. 
- Longer periods since previous purchase are associated with lower retention probability. 
- Customers who have been with the company longer tend to be more likely to remain retained.  

### Odds Ratios

Logistic Regression coefficients are converted into odds ratios to make
the magnitude of the relationships easier to interpret.

An odds ratio above 1 indicates higher odds of retention, while an odds
ratio below 1 indicates lower odds of retention.

In [11]:
import numpy as np

coefficient_df["Odds_Ratio"] = np.exp(
    coefficient_df["Coefficient"]
)

coefficient_df

,Feature,Coefficient,Odds_Ratio
0,Historical_Orders,1.254870,3.507384
1,Historical_Recency,-0.288053,0.749722
2,Customer_Tenure,0.238851,1.269789


Odds Ratio > 1 --> The feature is associated with higher odds of retention.

Odds Ratio < 1 --> The feature is associated with lower odds of retention.

Odds Ratio = 1 --> The feature has no change in the odds.

In [12]:
# Predicted Retention Probabilities
probability_summary = pd.Series(y_pred_proba).describe()
probability_summary

count    664.000000
mean       0.550695
std        0.199104
min        0.266349
25%        0.389990
50%        0.496118
75%        0.690727
max        1.000000
dtype: float64

### Customer-Level Retention Predictions

The resulting table allows individual customers to be identified by
their predicted retention probability and churn-risk classification.

In [13]:
test_predictions = retention_df.loc[
    X_test.index,
    [
        "CustomerID",
        "Historical_Orders",
        "Historical_Recency",
        "Customer_Tenure",
        "Retained"
    ]
].copy()

test_predictions["Predicted_Retention_Probability"] = y_pred_proba

test_predictions["Predicted_Retained"] = y_pred

test_predictions["Churn_Risk"] = (
    test_predictions["Predicted_Retained"] == 0
).astype(int)

test_predictions.head()

,CustomerID,Historical_Orders,Historical_Recency,Customer_Tenure,Retained,Predicted_Retention_Probability,Predicted_Retained,Churn_Risk
1574,15114.0,6,16,185,1,0.833696,1,0
1009,14116.0,1,21,0,1,0.483247,0,1
1552,15076.0,1,72,0,0,0.435725,0,1
182,12645.0,3,9,207,1,0.728358,1,0
1426,14847.0,1,48,0,0,0.457995,0,1


### Rank Customers by Churn Risk

Customers are ranked according to their predicted probability of
retention.

In [14]:
test_predictions = test_predictions.sort_values(
    "Predicted_Retention_Probability"
)

test_predictions.head()

,CustomerID,Historical_Orders,Historical_Recency,Customer_Tenure,Retained,Predicted_Retention_Probability,Predicted_Retained,Churn_Risk
1508,15012.0,1,273,0,1,0.266349,0,1
2217,16274.0,1,273,0,0,0.266349,0,1
3147,17968.0,1,273,0,0,0.266349,0,1
1287,14594.0,1,273,0,0,0.266349,0,1
1713,15350.0,1,273,0,0,0.266349,0,1


In [15]:
#  Retention Probability and Actual Outcome
probability_bins = [0, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 1.01]

test_predictions["Probability_Band"] = pd.cut(
    test_predictions["Predicted_Retention_Probability"],
    bins=probability_bins,
    right=False
)

risk_analysis = (
    test_predictions
    .groupby("Probability_Band", observed=False)
    .agg(
        Customers=("CustomerID", "count"),
        Actual_Churn_Rate=("Retained", lambda x: (x == 0).mean() * 100),
        Actual_Retention_Rate=("Retained", "mean")
    )
    .reset_index()
)

risk_analysis["Actual_Churn_Rate"] = risk_analysis["Actual_Churn_Rate"].round(2)
risk_analysis["Actual_Retention_Rate"] = risk_analysis["Actual_Retention_Rate"].mul(100).round(2)
risk_analysis

,Probability_Band,Customers,Actual_Churn_Rate,Actual_Retention_Rate
0,"[0.0, 0.3)",39,64.10,35.90
1,"[0.3, 0.4)",146,62.33,37.67
2,"[0.4, 0.5)",155,60.00,40.00
3,"[0.5, 0.6)",82,48.78,51.22
4,"[0.6, 0.7)",87,31.03,68.97
5,"[0.7, 0.8)",59,18.64,81.36
6,"[0.8, 1.01)",96,2.08,97.92


### Customer Risk Segmentation
Customers are grouped into three action-oriented risk categories based
on their predicted probability of retention.

- **High Risk**: predicted retention probability below 50%
- **Medium Risk**: predicted retention probability from 50% to below 70%
- **Low Risk**: predicted retention probability of 70% or higher

In [16]:
def assign_risk(probability):
    if probability < 0.50:
        return "High Risk"
    elif probability < 0.70:
        return "Medium Risk"
    else:
        return "Low Risk"


test_predictions["Risk_Category"] = test_predictions["Predicted_Retention_Probability"].apply(assign_risk)

risk_category_summary = (
    test_predictions
    .groupby("Risk_Category")
    .agg(
        Customers=("CustomerID", "count"),
        Avg_Retention_Probability=("Predicted_Retention_Probability", "mean"),
        Actual_Churn_Rate=("Retained", lambda x: (x == 0).mean() * 100)
    )
    .reset_index()
)

risk_category_summary["Avg_Retention_Probability"] = risk_category_summary["Avg_Retention_Probability"].round(2)
risk_category_summary["Actual_Churn_Rate"] = risk_category_summary["Actual_Churn_Rate"].round(2)

risk_category_summary

,Risk_Category,Customers,Avg_Retention_Probability,Actual_Churn_Rate
0,High Risk,340,0.39,61.47
1,Low Risk,155,0.85,8.39
2,Medium Risk,169,0.60,39.64


### Merge the existing Notebook 05 RFM segment onto the ML test predictions

In [17]:
rfm_file = PROCESSED_DIR / "customer_rfm.csv"

rfm_df = pd.read_csv(rfm_file)

print("RFM dataset:", rfm_df.shape)
rfm_df.head()

RFM dataset: (4338, 9)


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Segment
0,12346.0,325,1,77183.60,1,1,5,7,At Risk
1,12347.0,2,7,4310.00,5,5,5,15,Champions
2,12348.0,75,4,1797.24,2,4,4,10,At Risk
3,12349.0,18,1,1757.55,4,1,4,9,Recent Customers
4,12350.0,310,1,334.40,1,1,2,4,Hibernating / Low Engagement


In [18]:
# Keep only the RFM fields needed for comparison
rfm_segments = rfm_df[["CustomerID", "RFM_Segment"]].copy()

# Make sure CustomerID has the same type in both datasets
rfm_segments["CustomerID"] = rfm_segments["CustomerID"].astype(float)
test_predictions["CustomerID"] = test_predictions["CustomerID"].astype(float)

# Merge existing RFM segmentation with ML predictions
test_predictions = test_predictions.merge(
    rfm_segments,
    on="CustomerID",
    how="left"
)

print("RFM segments added successfully.")
print(test_predictions[["CustomerID", "RFM_Segment", "Risk_Category"]].head(10))

print("\nMissing RFM segments:",
      test_predictions["RFM_Segment"].isna().sum())

RFM segments added successfully.
   CustomerID                   RFM_Segment Risk_Category
0     15012.0                         Other     High Risk
1     16274.0  Hibernating / Low Engagement     High Risk
2     17968.0  Hibernating / Low Engagement     High Risk
3     14594.0              Recent Customers     High Risk
4     15350.0  Hibernating / Low Engagement     High Risk
5     17235.0              Recent Customers     High Risk
6     16781.0  Hibernating / Low Engagement     High Risk
7     12738.0  Hibernating / Low Engagement     High Risk
8     15545.0                         Other     High Risk
9     17967.0  Hibernating / Low Engagement     High Risk

Missing RFM segments: 0


In [19]:
# The existing RFM segments from Notebook 05 are compared with the Logistic Regression risk categories.
rfm_risk_table = pd.crosstab(
    test_predictions["RFM_Segment"],
    test_predictions["Risk_Category"]
)
rfm_risk_table

Risk_Category,High Risk,Low Risk,Medium Risk
RFM_Segment,,,
At Risk,62,15,79
Champions,27,107,40
Hibernating / Low Engagement,162,0,2
Loyal / Valuable,21,32,37
Other,37,1,6
Recent Customers,31,0,5


In [20]:
# Percentages within each RFM segment
rfm_risk_percentage = pd.crosstab(
    test_predictions["RFM_Segment"],
    test_predictions["Risk_Category"],
    normalize="index"
).mul(100).round(2)

rfm_risk_percentage

Risk_Category,High Risk,Low Risk,Medium Risk
RFM_Segment,,,
At Risk,39.74,9.62,50.64
Champions,15.52,61.49,22.99
Hibernating / Low Engagement,98.78,0.00,1.22
Loyal / Valuable,23.33,35.56,41.11
Other,84.09,2.27,13.64
Recent Customers,86.11,0.00,13.89


### Final Customer Retention Prediction Dataset

The final dataset combines historical customer behavior, the actual
retention outcome, Logistic Regression predictions, model-based risk
categories, and the existing RFM segment from Notebook 05.

In [23]:
final_predictions = test_predictions[
    [
        "CustomerID",
        "Historical_Orders",
        "Historical_Recency",
        "Customer_Tenure",
        "Retained",
        "Predicted_Retention_Probability",
        "Predicted_Retained",
        "Churn_Risk",
        "Risk_Category",
        "RFM_Segment"
    ]
].copy()
print("Final prediction dataset:", final_predictions.shape)
final_predictions.head()

Final prediction dataset: (664, 10)


,CustomerID,Historical_Orders,Historical_Recency,Customer_Tenure,Retained,Predicted_Retention_Probability,Predicted_Retained,Churn_Risk,Risk_Category,RFM_Segment
0,15012.0,1,273,0,1,0.266349,0,1,High Risk,Other
1,16274.0,1,273,0,0,0.266349,0,1,High Risk,Hibernating / Low Engagement
2,17968.0,1,273,0,0,0.266349,0,1,High Risk,Hibernating / Low Engagement
3,14594.0,1,273,0,0,0.266349,0,1,High Risk,Recent Customers
4,15350.0,1,273,0,0,0.266349,0,1,High Risk,Hibernating / Low Engagement


In [24]:
print("\nRisk category distribution:")
print(final_predictions["Risk_Category"].value_counts())

print("\nRFM segment distribution:")
print(final_predictions["RFM_Segment"].value_counts())


Risk category distribution:
Risk_Category
High Risk      340
Medium Risk    169
Low Risk       155
Name: count, dtype: int64

RFM segment distribution:
RFM_Segment
Champions                       174
Hibernating / Low Engagement    164
At Risk                         156
Loyal / Valuable                 90
Other                            44
Recent Customers                 36
Name: count, dtype: int64


In [25]:
# Save Final Retention Predictions
final_predictions_file = PROCESSED_DIR / "customer_retention_predictions.csv"

final_predictions.to_csv(
    final_predictions_file,
    index=False
)

print("Final retention prediction dataset saved to:")
print(final_predictions_file)

Final retention prediction dataset saved to:
C:\Users\tanya\Retail_Sales_Customer_Analytics\data\processed\customer_retention_predictions.csv
